In [0]:
caminho_volume = "/Volumes/workspace/default/raw/"
arquivos = dbutils.fs.ls(caminho_volume)

for a in arquivos:
    print(a.name, a.modificationTime)

In [0]:
arquivo_mais_novo = max(arquivos, key=lambda f: f.modificationTime)
print(arquivo_mais_novo.name)
print(arquivo_mais_novo.path)

In [0]:
import json

conteudo_raw = dbutils.fs.head(arquivo_mais_novo.path, 1000000)
dados = json.loads(conteudo_raw)

from_symbol = dados["Meta Data"]["2. From Symbol"]
to_symbol = dados["Meta Data"]["3. To Symbol"]
serie = dados["Time Series FX (Daily)"]

linhas = []
for data, valores in serie.items():
    linhas.append({
        "execution_date": data,
        "from_symbol": from_symbol,
        "to_symbol": to_symbol,
        "close_rate": float(valores["4. close"]),
    })

df_bronze_fx = spark.createDataFrame(linhas)
df_bronze_fx.show(10)
print(f"Total de linhas: {df_bronze_fx.count()}")

In [0]:
from pyspark.sql.functions import current_timestamp, lit

df_bronze_fx_final = (df_bronze_fx
    .withColumn("source_file", lit(arquivo_mais_novo.name))
    .withColumn("ingested_at", current_timestamp())
)

df_bronze_fx_final.write.mode("append").saveAsTable("workspace.default.bronze_fx_rates")

print("bronze_fx_rates atualizada com sucesso.")

In [0]:
%pip install --upgrade databricks-sdk>=0.81.0
%restart_python

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
endpoint_path = "projects/cambiotech-orders/branches/production/endpoints/primary"

credential = w.postgres.generate_database_credential(endpoint=endpoint_path)

print(f"Token gerado. Expira em: {credential.expire_time}")

In [0]:
host = "ep-wandering-tree-d8nxn0it.database.us-east-2.cloud.databricks.com"
port = "5432"
database = "databricks_postgres"
user = "gabrielgabridma@gmail.com"

url = f"jdbc:postgresql://{host}:{port}/{database}?sslmode=require"

df_orders = (spark.read
    .format("jdbc")
    .option("driver", "org.postgresql.Driver")
    .option("url", url)
    .option("dbtable", "public.orders")
    .option("user", user)
    .option("password", credential.token)
    .load()
)

df_orders.show(10, truncate=False)
print(f"Total de linhas: {df_orders.count()}")

In [0]:
from pyspark.sql.functions import current_timestamp

df_bronze_orders = df_orders.withColumn("ingested_at", current_timestamp())

df_bronze_orders.write.mode("overwrite").saveAsTable("workspace.default.bronze_orders")

print("bronze_orders atualizada com sucesso.")